### STEP 2. 실제 스키마·PK·FK를 먼저 검증

In [2]:
from pathlib import Path

# 노트북이 파일을 찾을 때 기준으로 사용하는 폴더 확인
print("현재 작업 폴더:", Path.cwd())

# 현재 폴더에 있는 CSV 파일 확인
print("CSV 파일:", [file.name for file in Path.cwd().glob("*.csv")])

현재 작업 폴더: c:\dev\llm-data-analysis-course\chapter12
CSV 파일: []


In [4]:
import pandas as pd
from pathlib import Path

# 한 단계 위로 이동한 뒤 전처리 데이터 폴더 지정
data_dir = Path("../data/processed")

# CSV 파일을 읽어 데이터프레임 만들기
customers = pd.read_csv(data_dir / "customers_clean.csv")
products = pd.read_csv(data_dir / "products_cleaned.csv")
orders = pd.read_csv(data_dir / "orders_clean.csv")
order_items = pd.read_csv(data_dir / "order_items_clean.csv")

# 불러온 데이터의 행 수와 열 수 확인
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)

customers: (150, 6)
products: (100, 4)
orders: (300, 7)
order_items: (752, 6)


In [5]:
# 불러온 데이터프레임을 이름별로 묶기
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

# 이번 검증에서 확인할 필수 컬럼
required_columns = {
    "customers": [
        "customer_id", "name", "gender", "age", "city", "signup_date"
    ],
    "products": [
        "product_id", "product_name", "category", "price"
    ],
    "orders": [
        "order_id", "customer_id", "order_date",
        "payment_method", "order_status"
    ],
    "order_items": [
        "order_item_id", "order_id", "product_id",
        "quantity", "unit_price", "line_total"
    ],
}

all_passed = True

# 각 데이터에서 빠진 필수 컬럼 찾기
for name, columns in required_columns.items():
    missing = [
        column for column in columns
        if column not in datasets[name].columns
    ]

    if missing:
        print(f"{name}: FAIL — 누락된 컬럼 {missing}")
        all_passed = False
    else:
        print(f"{name}: PASS — 필수 컬럼 모두 존재")

# 하나라도 실패하면 다음 검증으로 넘어가지 않기
assert all_passed, "필수 컬럼을 확인하고 다시 실행하세요."

print("\n성공 기준 ① 통과")

customers: PASS — 필수 컬럼 모두 존재
products: PASS — 필수 컬럼 모두 존재
orders: PASS — 필수 컬럼 모두 존재
order_items: PASS — 필수 컬럼 모두 존재

성공 기준 ① 통과


In [6]:
# 각 테이블의 PK 컬럼
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

all_passed = True

# PK 컬럼에 비어 있는 값이 있는지 확인
for name, pk in primary_keys.items():
    missing_count = datasets[name][pk].isna().sum()

    if missing_count == 0:
        print(f"{name}.{pk}: PASS — 결측 0건")
    else:
        print(f"{name}.{pk}: FAIL — 결측 {missing_count}건")
        all_passed = False

# 결측이 있으면 다음 단계로 넘어가지 않기
assert all_passed, "PK 결측값을 확인하고 다시 실행하세요."

print("\n성공 기준 ② 통과")

customers.customer_id: PASS — 결측 0건
products.product_id: PASS — 결측 0건
orders.order_id: PASS — 결측 0건
order_items.order_item_id: PASS — 결측 0건

성공 기준 ② 통과


In [7]:
all_passed = True

# 앞에서 정의한 primary_keys로 PK 중복 확인
for name, pk in primary_keys.items():
    duplicate_count = datasets[name][pk].duplicated().sum()

    if duplicate_count == 0:
        print(f"{name}.{pk}: PASS — 중복 0건")
    else:
        print(f"{name}.{pk}: FAIL — 중복 {duplicate_count}건")
        all_passed = False

# 중복이 있으면 다음 단계로 넘어가지 않기
assert all_passed, "PK 중복값을 확인하고 다시 실행하세요."

print("\n성공 기준 ③ 통과")

customers.customer_id: PASS — 중복 0건
products.product_id: PASS — 중복 0건
orders.order_id: PASS — 중복 0건
order_items.order_item_id: PASS — 중복 0건

성공 기준 ③ 통과


In [8]:
# 연결에 사용되는 부모 테이블과 키
parent_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
}

all_passed = True

# 부모 키의 모든 값이 서로 다른지 확인
for name, key in parent_keys.items():
    is_unique = datasets[name][key].is_unique

    if is_unique:
        print(f"{name}.{key}: PASS — 부모 키 고유")
    else:
        print(f"{name}.{key}: FAIL — 부모 키 중복")
        all_passed = False

# 부모 키가 중복되면 다음 단계로 넘어가지 않기
assert all_passed, "부모 키 중복을 확인하고 다시 실행하세요."

print("\n성공 기준 ④ 통과")

customers.customer_id: PASS — 부모 키 고유
orders.order_id: PASS — 부모 키 고유
products.product_id: PASS — 부모 키 고유

성공 기준 ④ 통과


In [9]:
# 각 테이블에서 확인할 FK 컬럼
foreign_keys = {
    "orders": ["customer_id"],
    "order_items": ["order_id", "product_id"],
}

all_passed = True

# FK 컬럼에 비어 있는 값이 있는지 확인
for name, keys in foreign_keys.items():
    for key in keys:
        missing_count = datasets[name][key].isna().sum()

        if missing_count == 0:
            print(f"{name}.{key}: PASS — FK 결측 0건")
        else:
            print(f"{name}.{key}: FAIL — FK 결측 {missing_count}건")
            all_passed = False

# FK 결측이 있으면 다음 단계로 넘어가지 않기
assert all_passed, "FK 결측값을 확인하고 다시 실행하세요."

print("\n성공 기준 ⑤ 통과")

orders.customer_id: PASS — FK 결측 0건
order_items.order_id: PASS — FK 결측 0건
order_items.product_id: PASS — FK 결측 0건

성공 기준 ⑤ 통과


In [11]:
# 앞에서 정의한 relationships를 사용
for child, fk, parent, pk in relationships:
    # 부모 테이블에 없는 FK 찾기
    unmatched = ~datasets[child][fk].isin(datasets[parent][pk])

    print(f"\n{child}.{fk} → {parent}.{pk}")
    print(f"미매칭 행 수: {unmatched.sum()}건")

    # 행 구분용 PK와 문제가 있는 FK만 표시
    child_pk = primary_keys[child]
    display(datasets[child].loc[unmatched, [child_pk, fk]])


orders.customer_id → customers.customer_id
미매칭 행 수: 2건


,order_id,customer_id
125,126,9991
277,278,9992



order_items.order_id → orders.order_id
미매칭 행 수: 2건


,order_item_id,order_id
147,150,9991
609,620,9992



order_items.product_id → products.product_id
미매칭 행 수: 2건


,order_item_id,product_id
256,260,9991
690,701,9992


In [13]:
# 원본을 유지하고 검증용 복사본 만들기
validated_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

# 실제 고객이 존재하는 주문만 선택
valid_orders = orders[
    orders["customer_id"].isin(customers["customer_id"])
].copy()

# 남은 주문과 실제 상품을 모두 참조하는 주문 상세만 선택
valid_items = order_items[
    order_items["order_id"].isin(valid_orders["order_id"])
    & order_items["product_id"].isin(products["product_id"])
].copy()

# 검증용 데이터에 반영
validated_datasets["orders"] = valid_orders
validated_datasets["order_items"] = valid_items

# 제외한 행 수 기록
print(f"orders: {len(orders)} → {len(valid_orders)}행")
print(f"제외한 주문: {len(orders) - len(valid_orders)}행")

print(f"\norder_items: {len(order_items)} → {len(valid_items)}행")
print(f"제외한 주문 상세: {len(order_items) - len(valid_items)}행")

orders: 300 → 298행
제외한 주문: 2행

order_items: 752 → 745행
제외한 주문 상세: 7행


In [14]:
# 검증 결과를 모을 목록
results = []

# ① 필수 컬럼 확인
for name, columns in required_columns.items():
    missing = [
        column for column in columns
        if column not in validated_datasets[name].columns
    ]
    results.append(("① 필수 컬럼", name, len(missing)))

# 필수 컬럼이 없으면 이후 검증 중단
assert all(row[2] == 0 for row in results), "필수 컬럼이 누락됐습니다."

# ② PK 결측, ③ PK 중복 확인
for name, pk in primary_keys.items():
    values = validated_datasets[name][pk]

    results.append(("② PK 결측", f"{name}.{pk}", values.isna().sum()))
    results.append(("③ PK 중복", f"{name}.{pk}", values.duplicated().sum()))

# ④ 부모 키 고유성 확인: 중복이 0건이면 고유함
for name, key in parent_keys.items():
    duplicates = validated_datasets[name][key].duplicated().sum()
    results.append(("④ 부모 키 고유성", f"{name}.{key}", duplicates))

# ⑤ FK 결측, ⑥ 부모에 없는 FK 확인
for child, fk, parent, pk in relationships:
    child_values = validated_datasets[child][fk]
    parent_values = validated_datasets[parent][pk]

    # 결측은 ⑤에서 확인하므로 ⑥에서는 제외하고 검사
    unmatched = ~child_values.dropna().isin(parent_values)

    results.append(("⑤ FK 결측", f"{child}.{fk}", child_values.isna().sum()))
    results.append(("⑥ FK 미매칭", f"{child}.{fk}", unmatched.sum()))

# 결과를 표로 표시
check_result = pd.DataFrame(
    results,
    columns=["성공 기준", "검증 대상", "문제 건수"]
)
check_result["결과"] = [
    "PASS" if count == 0 else "FAIL"
    for count in check_result["문제 건수"]
]

display(check_result)

# 모든 항목이 통과했는지 확인
assert check_result["문제 건수"].eq(0).all(), "실패한 항목을 확인하세요."

print("검증용 복사본: 성공 기준 6가지 모두 통과")

,성공 기준,검증 대상,문제 건수,결과
0,① 필수 컬럼,customers,0,PASS
1,① 필수 컬럼,products,0,PASS
2,① 필수 컬럼,orders,0,PASS
3,① 필수 컬럼,order_items,0,PASS
4,② PK 결측,customers.customer_id,0,PASS
5,③ PK 중복,customers.customer_id,0,PASS
6,② PK 결측,products.product_id,0,PASS
7,③ PK 중복,products.product_id,0,PASS
8,② PK 결측,orders.order_id,0,PASS
9,③ PK 중복,orders.order_id,0,PASS


검증용 복사본: 성공 기준 6가지 모두 통과


### STEP 2. 실제 스키마·PK·FK 검증 — 결과 확인

#### 결과 관찰

전처리 CSV 4개를 불러와 성공 기준 6가지를 검증했다.

| 성공 기준 | 최초 데이터 | 검증용 복사본 |
|---|---|---|
| ① 필수 컬럼 존재 | PASS | PASS |
| ② PK 결측 없음 | PASS | PASS |
| ③ PK 중복 없음 | PASS | PASS |
| ④ 부모 키 고유성 | PASS | PASS |
| ⑤ FK 결측 없음 | PASS | PASS |
| ⑥ 부모 테이블에 없는 FK 없음 | FAIL | PASS |

최초 검증에서는 다음 관계에서 각각 미매칭 2건이 발견됐다. 부모 테이블에 없는 ID는 모두 `9991`, `9992`였다.

- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`

#### 나의 해석과 판단

FK에 결측이 없더라도 해당 ID가 부모 테이블에 존재한다는 뜻은 아니다. 최초 데이터는 FK 미매칭으로 성공 기준 ⑥을 충족하지 못했으며, `AssertionError`로 다음 단계 진행이 중단됐다.

문제 행을 제외한 검증용 복사본에서는 성공 기준 6가지를 모두 통과했다.

#### 수정 내용

원본 CSV와 기존 `datasets`를 유지하고, 검증용 복사본 `validated_datasets`를 만들었다.

실제 고객을 참조하는 주문만 남긴 뒤, 남은 주문과 실제 상품을 모두 참조하는 주문 상세만 선택했다.

| 데이터 | 처리 전 | 처리 후 | 제외한 행 |
|---|---:|---:|---:|
| orders | 300 | 298 | 2 |
| order_items | 752 | 745 | 7 |

#### Evidence

최초 FK 검증의 미매칭 출력, 문제 ID 조회 결과, 제외 전후 행 수를 확인했다. 복사본 재검증 결과표의 21개 검증 항목은 모두 문제 건수 0건, PASS로 나타났다.

#### 한계와 추가 확인 사항

이번 처리는 올바른 ID를 복구한 것이 아니라, 관계를 확인할 수 없는 행을 분석 대상에서 제외한 것이다. 따라서 이후 분석 결과는 제외 후 데이터 범위에 해당하며, 제외로 인한 집계 금액 변화도 확인해야 한다.

현재 확인한 범위는 필수 컬럼과 PK/FK 관계다. 금액 계산식, merge 전후 행 수, completed 주문 범위와 집계 총합은 STEP 3에서 추가 검증한다.

### STEP 3. merge와 completed 집계의 Business Rule 검증

In [16]:
import pandas as pd
import numpy as np

# STEP 2 데이터는 보존하고 별도 복사본 사용
orders_base = validated_datasets["orders"].copy()
items_base = validated_datasets["order_items"].copy()
products_check = validated_datasets["products"].copy()


# 1. 날짜 확인 및 집계 범위 설정
parsed_dates = pd.to_datetime(
    orders_base["order_date"],
    errors="coerce",
)

invalid_dates = parsed_dates.isna()

print(f"날짜 결측·변환 실패 주문: {invalid_dates.sum()}건")
display(
    orders_base.loc[
        invalid_dates,
        ["order_id", "order_date", "order_status"]
    ]
)

# 날짜가 확인되는 주문만 선택
orders_check = orders_base.loc[~invalid_dates].copy()
orders_check["order_date"] = parsed_dates.loc[~invalid_dates]

# 남은 주문에 연결된 주문 상세만 선택
items_check = items_base[
    items_base["order_id"].isin(orders_check["order_id"])
].copy()

print(f"주문: {len(orders_base)} → {len(orders_check)}행")
print(f"주문 상세: {len(items_base)} → {len(items_check)}행")


# 2. 부모 키와 금액 계산식 확인
assert orders_check["order_id"].is_unique, "주문 ID 중복"
assert products_check["product_id"].is_unique, "상품 ID 중복"

# 금액 관련 값에 결측이나 무한대가 없는지 확인
amount_columns = ["quantity", "unit_price", "line_total"]

assert np.isfinite(
    items_check[amount_columns].to_numpy(dtype=float)
).all(), "금액 관련 컬럼에 결측 또는 무한대가 있습니다."

expected_amount = (
    items_check["quantity"] * items_check["unit_price"]
)

amount_error_count = (
    ~np.isclose(
        items_check["line_total"],
        expected_amount,
        rtol=0,
        atol=0.01,
    )
).sum()

assert amount_error_count == 0, "line_total 계산식 불일치"


# 3. 상품 및 주문 연결
before_rows = len(items_check)

merged = items_check.merge(
    products_check[["product_id", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

product_rows = len(merged)
product_unmatched = merged["_merge"].ne("both").sum()
merged = merged.drop(columns="_merge")

merged = merged.merge(
    orders_check[["order_id", "order_status", "order_date"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

order_rows = len(merged)
order_unmatched = merged["_merge"].ne("both").sum()
merged = merged.drop(columns="_merge")

assert product_rows == before_rows, "상품 merge 행 수 불일치"
assert order_rows == before_rows, "주문 merge 행 수 불일치"
assert product_unmatched == 0, "상품 미매칭 발견"
assert order_unmatched == 0, "주문 미매칭 발견"

category_values = merged["category"].astype("string")
category_missing = (
    category_values.isna()
    | category_values.str.strip().eq("").fillna(False)
).sum()

assert category_missing == 0, "카테고리 결측 또는 빈 값 발견"


# 4. 날짜가 있는 completed 주문의 기준 총합 계산
completed_ids = orders_check.loc[
    orders_check["order_status"].eq("completed"),
    "order_id",
]

# 기준 총합은 merge 전 주문 상세에서 계산
source_items = items_check[
    items_check["order_id"].isin(completed_ids)
]
source_total = source_items["line_total"].sum()

completed = merged[
    merged["order_status"].eq("completed")
].copy()

assert len(completed) > 0, "집계 대상이 없습니다."
assert len(completed) == len(source_items), "completed 범위 행 수 불일치"

print(f"\n집계 대상 completed 주문: {len(completed_ids)}건")
print(f"집계 대상 completed 주문 상세: {len(completed)}행")


# 5. 카테고리별 집계
category_sales = (
    completed.groupby("category", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "total_sales"})
    .sort_values("total_sales", ascending=False)
    .reset_index(drop=True)
)

category_total = category_sales["total_sales"].sum()

print("\n카테고리별 완료 주문 상세 금액")
display(category_sales)


# 6. 월별 집계
completed["order_month"] = (
    completed["order_date"]
    .dt.to_period("M")
    .astype(str)
)

monthly_sales = (
    completed.groupby("order_month", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "total_sales"})
    .sort_values("order_month")
    .reset_index(drop=True)
)

monthly_total = monthly_sales["total_sales"].sum()

print("\n월별 완료 주문 상세 금액")
display(monthly_sales)


# 7. 날짜 제외에 따른 completed 금액 변화 기록
before_completed_ids = orders_base.loc[
    orders_base["order_status"].eq("completed"),
    "order_id",
]

before_total = items_base.loc[
    items_base["order_id"].isin(before_completed_ids),
    "line_total",
].sum()

scope_change = pd.DataFrame({
    "구분": [
        "날짜 제외 전 completed 금액",
        "날짜가 있는 completed 금액",
        "날짜 결측으로 제외한 completed 금액",
    ],
    "금액": [
        before_total,
        source_total,
        before_total - source_total,
    ],
})

print("\n집계 범위 변경 기록")
display(scope_change)


# 8. 검증 결과 및 총합 대조
validation_result = pd.DataFrame({
    "검증 항목": [
        "line_total 불일치",
        "상품 merge 행 수 차이",
        "주문 merge 행 수 차이",
        "상품 미매칭",
        "주문 미매칭",
        "카테고리 결측 또는 빈 값",
        "집계 대상 날짜 결측",
    ],
    "문제 건수": [
        amount_error_count,
        product_rows - before_rows,
        order_rows - before_rows,
        product_unmatched,
        order_unmatched,
        category_missing,
        completed["order_date"].isna().sum(),
    ],
})

validation_result["결과"] = [
    "PASS" if count == 0 else "FAIL"
    for count in validation_result["문제 건수"]
]

print("\n검증 결과")
display(validation_result)

total_check = pd.DataFrame({
    "구분": ["기준 총합", "카테고리 총합", "월별 총합"],
    "금액": [source_total, category_total, monthly_total],
})

print("\n동일한 집계 범위의 총합 대조")
display(total_check)

assert validation_result["문제 건수"].eq(0).all(), "검증 실패"

assert np.isclose(
    source_total, category_total, rtol=0, atol=0.01
), "카테고리 총합 불일치"

assert np.isclose(
    source_total, monthly_total, rtol=0, atol=0.01
), "월별 총합 불일치"

print("\nSTEP 3: 날짜가 있는 completed 주문 범위에서 검증 통과")

날짜 결측·변환 실패 주문: 3건


,order_id,order_date,order_status
71,72,NaN,completed
142,143,NaN,refunded
213,214,NaN,completed


주문: 298 → 295행
주문 상세: 745 → 736행

집계 대상 completed 주문: 177건
집계 대상 completed 주문 상세: 438행

카테고리별 완료 주문 상세 금액


,category,total_sales
0,생활용품,27843000.0
1,전자기기,26727000.0
2,스포츠,22628000.0
3,도서,20554000.0
4,뷰티,16954000.0
5,패션,14932000.0
6,식품,9347000.0



월별 완료 주문 상세 금액


...


검증 결과


,검증 항목,문제 건수,결과
0,line_total 불일치,0,PASS
1,상품 merge 행 수 차이,0,PASS
2,주문 merge 행 수 차이,0,PASS
3,상품 미매칭,0,PASS
4,주문 미매칭,0,PASS
5,카테고리 결측 또는 빈 값,0,PASS
6,집계 대상 날짜 결측,0,PASS



동일한 집계 범위의 총합 대조


,구분,금액
0,기준 총합,138985000.0
1,카테고리 총합,138985000.0
2,월별 총합,138985000.0



STEP 3: 날짜가 있는 completed 주문 범위에서 검증 통과


### STEP 3. merge와 completed 집계 검증 — 결과 확인

#### 결과 관찰

`line_total = quantity × unit_price` 불일치는 0건이었다. 상품과 주문을 연결할 때 `validate="many_to_one"`을 적용했으며, merge 전후 행 수 차이와 미매칭도 모두 0건이었다. 카테고리 결측 또는 빈 값은 없었다.

최초 날짜 검증에서 주문 3건의 날짜 결측을 발견했다.

- 주문 `72`, `214`: completed
- 주문 `143`: refunded

날짜가 확인되는 주문만 선택하고, 연결된 주문 상세도 같은 범위로 제한했다.

| 항목 | 처리 전 | 처리 후 |
|---|---:|---:|
| 주문 | 298행 | 295행 |
| 주문 상세 | 745행 | 736행 |
| completed 주문 | 179건 | 177건 |
| completed 주문 상세 | 444행 | 438행 |

#### 나의 해석과 판단

날짜가 없는 completed 주문은 월별 집계에 포함할 수 없으므로, 카테고리 집계에서도 동일하게 제외했다. 집계 범위를 일치시킨 뒤 기준 총합과 두 집계 총합이 같은지 확인했다.

검증 결과, **날짜가 있는 completed 주문 범위에서 STEP 3의 검사 항목을 모두 통과했다.**

#### Evidence

| 총합 대조 항목 | 금액 |
|---|---:|
| merge 전 주문 상세에서 계산한 기준 총합 | 138,985,000 |
| 카테고리 집계 총합 | 138,985,000 |
| 월별 집계 총합 | 138,985,000 |

금액 계산, merge 행 수, 미매칭, 카테고리 결측, 집계 대상 날짜 결측에 대한 7개 검사 항목은 모두 문제 건수 0건, PASS였다.

#### 수정 내용

기존 `validated_datasets`를 보존하고 STEP 3용 복사본을 사용했다. 날짜가 없는 주문 3행과 연결된 주문 상세 9행을 제외했다.

이 중 completed 주문 상세의 제외 금액은 **1,903,000**이었다. 따라서 completed 금액은 140,888,000에서 138,985,000으로 감소했다.

#### 한계와 추가 확인 사항

날짜 결측을 복구한 것이 아니라 집계 범위를 제한했으므로, 결과는 날짜가 확인되는 completed 주문에만 해당한다.

월별 결과에 `2027-01` 금액 909,000이 포함되어 있다. 현재 날짜인 2026-10-06 이후의 날짜이므로, 실습 데이터의 허용 기간에 맞는지 추가 확인이 필요하다. 날짜 변환 성공만으로 날짜의 업무상 타당성까지 확인한 것은 아니다.

계산한 금액은 완료 주문 상세 금액 합계다. 할인·배송비·세금·환불 정산을 반영한 회계상 순매출로 해석하지 않는다.

### STEP 4. Generated Code를 실행하지 않고 AST Static Scan

In [18]:
import sys
from pathlib import Path

# 현재 chapter12 폴더의 한 단계 위가 프로젝트 루트
project_root = Path.cwd().parent

# 공식 검증 모듈이 있는지 확인
module_path = project_root / "src" / "llm_code_validation.py"
assert module_path.is_file(), f"검증 모듈을 찾을 수 없습니다: {module_path}"

# 노트북에서 프로젝트의 src 모듈을 불러올 수 있도록 설정
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.llm_code_validation import (
    DEFAULT_STATIC_SCAN_EXAMPLE,
    scan_generated_code,
)

# 검증 대상은 실행할 코드가 아니라 코드가 담긴 문자열
generated_code = DEFAULT_STATIC_SCAN_EXAMPLE

print("검증 대상 코드 — 읽기만 수행")
print(generated_code)

# 코드를 실행하지 않고 AST로 정적 검사
static_scan = scan_generated_code(generated_code)

print("\nAST 정적 스캔 결과")
display(static_scan)

print("\n정적 스캔 완료 — 검증 대상 코드는 실행하지 않았습니다.")

검증 대상 코드 — 읽기만 수행
import requests

response = requests.get("https://example.com/data.csv")
open("download.csv", "wb").write(response.content)

AST 정적 스캔 결과


,severity,category,line,detail
0,review,import,1,외부 작업 가능 모듈 import: requests
1,high,network,3,외부 URL 사용: requests.get
2,high,operation,3,외부 네트워크 요청: requests.get
3,high,file_write,4,"파일 쓰기 모드: open(..., 'wb')"



정적 스캔 완료 — 검증 대상 코드는 실행하지 않았습니다.


### STEP 4. AST 정적 스캔 — 결과 확인

#### 결과 요약

공식 예제 코드를 문자열로 읽고 AST 정적 스캔을 수행했다. 검증 대상 코드는 실행하지 않았다.

총 4건의 검토·위험 항목이 탐지됐다.

| 심각도 | 유형 | 코드 위치 | 탐지 내용 |
|---|---|---|---|
| review | import | 1번 줄 | 외부 통신에 사용할 수 있는 requests 모듈 불러오기 |
| high | network | 3번 줄 | requests.get에서 외부 URL 사용 |
| high | operation | 3번 줄 | 외부 네트워크 요청 |
| high | file_write | 4번 줄 | open(..., "wb")로 파일 생성 또는 덮어쓰기 |

`review`는 1건, `high`는 3건이었다.

3번 줄의 동일한 요청이 ‘외부 URL 사용’과 ‘네트워크 요청’으로 각각 탐지됐다. 따라서 탐지 4건이 서로 다른 작업 4개를 의미하는 것은 아니다.

#### 쉽게 발생할 수 있는 위험 사례

| 위험 사례 | 실생활 비유 | 코드에서 발생할 수 있는 문제 |
|---|---|---|
| 기존 파일 덮어쓰기 | 과제를 같은 파일명으로 저장해 이전 내용이 사라짐 | 같은 위치에 download.csv가 있으면 "wb" 모드가 기존 내용을 지운다. |
| 엉뚱한 자료 저장 | 교재를 주문했는데 광고 전단지가 배달됨 | CSV 대신 로그인 화면이나 오류 페이지를 받아도 download.csv라는 이름으로 저장할 수 있다. |
| 민감정보 외부 전달 | 배달 주소란에 현관 비밀번호까지 적어 보냄 | URL이나 요청 헤더에 개인정보·토큰을 넣으면 외부 서버로 전달된다. 현재 예제에는 그런 정보가 보이지 않는다. |
| 응답을 오래 기다림 | 상대가 응답하지 않는 전화를 계속 기다림 | timeout을 지정하지 않아 서버 응답이 지연되면 실행이 오래 멈출 수 있다. |

위 사례는 발생 가능한 문제를 설명한 것이다. 이번 실습에서 파일 손실이나 정보 유출이 실제로 발생한 것은 아니다.

#### 나의 해석과 판단

`import requests`는 외부 통신 기능을 사용할 준비이며, import 자체가 요청을 보내는 것은 아니다.

반면 `requests.get(...)`은 실제 외부 요청이고, `open(..., "wb")`는 파일을 생성하거나 덮어쓰는 작업이다. 실행 전에 요청 목적·대상 URL·저장 경로·기존 파일 존재 여부를 확인해야 한다.

현재 예제에는 high 항목이 있으므로 **현재 상태의 실행 판단은 BLOCK**으로 기록한다. 정적 스캔 완료는 실행 승인을 의미하지 않는다.

#### Evidence와 수정 내용

Evidence는 검증 대상 코드 문자열, AST 스캔 결과표, 검증 대상 코드를 실행하지 않았다는 출력이다.

이번 단계에서는 예제 코드를 수정하지 않고 탐지 결과를 확인했다. 외부 요청과 파일 저장도 실행하지 않았다.

#### 한계와 추가 확인 사항

정적 스캔은 코드 구조에서 위험 패턴을 찾는 검사다. 서버가 어떤 내용을 반환하는지, 기존 파일에 중요한 내용이 있는지까지 확인해 주지는 않는다.

탐지 결과가 0건이어도 코드가 안전하다고 단정할 수 없다. 분석 논리, 입력·출력 범위, 실행 환경을 사람이 함께 검토해야 한다.

### STEP 5. 회귀와 분류의 Feature Contract를 따로 검토
Chapter 09 회귀 

대표 금지 Feature:

order_total
line_total
quantity
unit_price
item_count
total_quantity
avg_unit_price
order_status
order_id
customer_id

Chapter 10 분류

대표 금지 Feature:

order_status
is_cancelled
order_id
customer_id
product_id
row-level line_total
row-level quantity
row-level unit_price
cancel_reason
cancelled_at

In [19]:
from src.llm_code_validation import (
    build_leakage_review_table,
    build_feature_audit,
    validate_feature_list,
)

# 1. 회귀·분류의 Feature 사용 기준 확인
leakage_contract = build_leakage_review_table()

print("회귀·분류 Feature Contract")
display(leakage_contract)


# 2. 회귀: 주문 금액 예측에 사용할 Feature 후보
regression_features = [
    "payment_method",
    "order_month",
    "order_dayofweek",
    "gender",
    "age",
    "city",
]

print("\n회귀 Feature 후보")
print(regression_features)

# 금지된 Feature가 포함됐는지 검사
validate_feature_list(
    regression_features,
    problem="regression",
)

regression_audit = build_feature_audit(
    regression_features,
    problem="regression",
)

print("\n회귀 Feature 검토 결과")
display(regression_audit)


# 3. 분류: 주문 취소 예측에 사용할 Feature 후보
classification_features = [
    "payment_method",
    "item_count",
    "total_quantity",
    "order_amount",
    "age",
    "city",
]

print("\n분류 Feature 후보")
print(classification_features)

validate_feature_list(
    classification_features,
    problem="classification",
)

classification_audit = build_feature_audit(
    classification_features,
    problem="classification",
)

print("\n분류 Feature 검토 결과")
display(classification_audit)

print("\nFeature 목록 검사 완료 — 모델 학습은 수행하지 않았습니다.")
print("REVIEW 항목은 예측 시점에 값을 알 수 있는지 추가 확인해야 합니다.")

회귀·분류 Feature Contract


,problem,prediction_time,target,forbidden_examples,split_rule
0,regression,주문 상세 기반 target 재료를 아직 사용할 수 없는 시점,order_total,"avg_unit_price, customer_id, item_count, line_...",날짜 순서 split + train 내부 TimeSeriesSplit selecti...
1,classification,주문 생성 직후,"completed=0, cancelled=1; refunded/other 제외","cancel_reason, cancelled_at, customer_id, is_c...",교육용 stratified train/validation/test; model/th...



회귀 Feature 후보
['payment_method', 'order_month', 'order_dayofweek', 'gender', 'age', 'city']

회귀 Feature 검토 결과


,problem,feature,status,review_note
0,regression,payment_method,REVIEW,예측 시점 가용성을 사람이 확인
1,regression,order_month,REVIEW,예측 시점 가용성을 사람이 확인
2,regression,order_dayofweek,REVIEW,예측 시점 가용성을 사람이 확인
3,regression,gender,REVIEW,예측 시점 가용성을 사람이 확인
4,regression,age,REVIEW,예측 시점 가용성을 사람이 확인
5,regression,city,REVIEW,예측 시점 가용성을 사람이 확인



분류 Feature 후보
['payment_method', 'item_count', 'total_quantity', 'order_amount', 'age', 'city']

분류 Feature 검토 결과


,problem,feature,status,review_note
0,classification,payment_method,REVIEW,예측 시점 가용성을 사람이 확인
1,classification,item_count,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
2,classification,total_quantity,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
3,classification,order_amount,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
4,classification,age,REVIEW,예측 시점 가용성을 사람이 확인
5,classification,city,REVIEW,예측 시점 가용성을 사람이 확인



Feature 목록 검사 완료 — 모델 학습은 수행하지 않았습니다.
REVIEW 항목은 예측 시점에 값을 알 수 있는지 추가 확인해야 합니다.


### STEP 5. 회귀·분류 Feature Contract 검토 — 결과 확인

#### 결과 관찰

회귀와 분류의 Feature Contract를 확인하고, 각 문제의 Feature 후보 6개를 검사했다.

| 문제 | Feature 후보 수 | 금지 Feature 포함 | 검토 결과 |
|---|---:|---|---|
| 회귀 | 6개 | 없음 | 모두 REVIEW |
| 분류 | 6개 | 없음 | 모두 REVIEW |

회귀 후보는 payment_method, order_month, order_dayofweek, gender, age, city였다.

분류 후보는 payment_method, item_count, total_quantity, order_amount, age, city였다.

Feature 목록 검사는 오류 없이 완료됐으며, 모델 학습은 수행하지 않았다.

#### 나의 해석과 판단

금지된 Feature가 없다는 것은 해당 Feature를 즉시 사용해도 된다는 뜻이 아니다. 실제 예측 시점에 값을 알 수 있었는지 사람이 확인해야 한다.

| 확인할 항목 | 검토 내용 |
|---|---|
| payment_method | 예측 시점에 결제수단이 이미 선택되어 있었는가? |
| order_month, order_dayofweek | 예측 시점에 알고 있는 주문 날짜로 계산했는가? |
| gender, age, city | 예측 시점에 확보한 고객 정보이며, 이후 변경된 값을 사용하지 않았는가? |
| item_count, total_quantity, order_amount | 주문 생성 시 확정된 값이며, 취소·환불 후 수정된 값이 아닌가? |

특히 분류의 주문 집계 Feature 3개는 **주문 생성 시 상품 구성·수량·금액이 확정된다는 교육용 가정**을 확인해야 한다.

현재는 가용성을 입증하는 기록을 확인하지 않았으므로 REVIEW 상태를 유지한다.

#### Evidence

Feature Contract 표와 회귀·분류 Feature Audit 표를 확인했다. 회귀 6개와 분류 6개 모두 REVIEW로 표시됐으며, 각 항목에 예측 시점 또는 교육용 가정의 확인 필요성이 기록됐다.

#### 수정 내용

이번 단계에서는 Feature 목록을 변경하지 않았다. 회귀와 분류의 후보를 따로 정의하고 각각의 계약으로 검사했다.

#### 한계와 추가 확인 사항

현재 검사는 Feature 이름을 계약과 비교한 것이다. 실제 값의 생성 시각, 변경 이력, 집계 과정까지 검증하지는 않았다.

Feature Leakage는 컬럼 이름만의 문제가 아니라 **예측 시점에 알 수 없는 정보를 미리 사용하는 문제**다. 실제 데이터를 구성할 때 각 값의 생성·변경 시점과 집계 범위를 추가 확인해야 한다.

자동 검토 결과인 REVIEW를 근거 없이 PASS로 변경하지 않는다.

### STEP 6. Sandbox와 Package 검토

In [20]:
import sys
import pandas as pd
from importlib.metadata import version


# 1. 현재 Python과 사용 중인 패키지 버전 확인
environment_info = pd.DataFrame({
    "항목": ["Python", "pandas", "numpy"],
    "현재 버전": [
        sys.version.split()[0],
        version("pandas"),
        version("numpy"),
    ],
})

print("현재 실행 환경")
display(environment_info)


# 2. Sandbox 검토표
sandbox_checklist = pd.DataFrame([
    ("입력 데이터", "복사한 소량 샘플을 사용하는가?"),
    ("입력 보호", "입력 파일에 읽기 전용 설정을 적용했는가?"),
    ("인증정보", "API Key·DB 비밀번호·클라우드 인증정보가 없는가?"),
    ("격리 환경", "작업 후 폐기할 수 있는 별도 환경인가?"),
    ("출력 경로", "파일 쓰기를 승인된 폴더로 제한했는가?"),
    ("네트워크", "외부 통신을 기본 차단했는가?"),
    ("자원 제한", "CPU·메모리·실행 시간·프로세스·디스크를 제한했는가?"),
    ("실행 전 기록", "기존 파일 목록·크기·해시를 기록했는가?"),
    ("실행 후 비교", "생성·수정·삭제된 파일을 비교할 계획이 있는가?"),
], columns=["검토 항목", "확인할 내용"])

# 실제 설정을 확인하기 전에는 PASS로 표시하지 않음
sandbox_checklist["상태"] = "REVIEW"
sandbox_checklist["Evidence"] = "설정 또는 기록 확인 필요"

print("\nSandbox 검토표")
display(sandbox_checklist)


# 3. 새 패키지 설치 검토표
package_review = pd.DataFrame([
    ("필요성", "기존 패키지로 해결할 수 없는가?"),
    ("이름과 출처", "공식 패키지 이름과 배포 출처가 맞는가?"),
    ("유사 이름", "유명 패키지와 이름이 비슷한 다른 패키지는 아닌가?"),
    ("버전 고정", "설치할 정확한 버전을 지정했는가?"),
    ("Python 호환성", "현재 Python 버전을 지원하는가?"),
    ("추가 의존성", "함께 설치되는 패키지를 검토했는가?"),
    ("설치 동작", "설치 과정에서 실행되는 코드를 검토했는가?"),
    ("설치 환경", "격리된 환경에 설치하는가?"),
    ("재현 기록", "requirements 또는 lock 파일에 기록하는가?"),
    ("정책", "수업 또는 조직의 설치 정책에 맞는가?"),
], columns=["검토 항목", "확인할 내용"])

package_review["상태"] = "REVIEW"
package_review["Evidence"] = "설치 대상 지정 후 확인 필요"

print("\n새 패키지 설치 검토표")
display(package_review)

package_decision = "DO_NOT_INSTALL_UNTIL_REVIEWED"

print(f"\n새 패키지 설치 결정: {package_decision}")
print("이번 셀에서는 패키지를 설치하지 않았습니다.")
print("Sandbox 준비 여부와 코드 실행 승인 여부는 별도로 확인해야 합니다.")

현재 실행 환경


,항목,현재 버전
0,Python,3.14.6
1,pandas,3.0.5
2,numpy,2.5.1



Sandbox 검토표


,검토 항목,확인할 내용,상태,Evidence
0,입력 데이터,복사한 소량 샘플을 사용하는가?,REVIEW,설정 또는 기록 확인 필요
1,입력 보호,입력 파일에 읽기 전용 설정을 적용했는가?,REVIEW,설정 또는 기록 확인 필요
2,인증정보,API Key·DB 비밀번호·클라우드 인증정보가 없는가?,REVIEW,설정 또는 기록 확인 필요
3,격리 환경,작업 후 폐기할 수 있는 별도 환경인가?,REVIEW,설정 또는 기록 확인 필요
4,출력 경로,파일 쓰기를 승인된 폴더로 제한했는가?,REVIEW,설정 또는 기록 확인 필요
5,네트워크,외부 통신을 기본 차단했는가?,REVIEW,설정 또는 기록 확인 필요
6,자원 제한,CPU·메모리·실행 시간·프로세스·디스크를 제한했는가?,REVIEW,설정 또는 기록 확인 필요
7,실행 전 기록,기존 파일 목록·크기·해시를 기록했는가?,REVIEW,설정 또는 기록 확인 필요
8,실행 후 비교,생성·수정·삭제된 파일을 비교할 계획이 있는가?,REVIEW,설정 또는 기록 확인 필요



새 패키지 설치 검토표


,검토 항목,확인할 내용,상태,Evidence
0,필요성,기존 패키지로 해결할 수 없는가?,REVIEW,설치 대상 지정 후 확인 필요
1,이름과 출처,공식 패키지 이름과 배포 출처가 맞는가?,REVIEW,설치 대상 지정 후 확인 필요
2,유사 이름,유명 패키지와 이름이 비슷한 다른 패키지는 아닌가?,REVIEW,설치 대상 지정 후 확인 필요
3,버전 고정,설치할 정확한 버전을 지정했는가?,REVIEW,설치 대상 지정 후 확인 필요
4,Python 호환성,현재 Python 버전을 지원하는가?,REVIEW,설치 대상 지정 후 확인 필요
5,추가 의존성,함께 설치되는 패키지를 검토했는가?,REVIEW,설치 대상 지정 후 확인 필요
6,설치 동작,설치 과정에서 실행되는 코드를 검토했는가?,REVIEW,설치 대상 지정 후 확인 필요
7,설치 환경,격리된 환경에 설치하는가?,REVIEW,설치 대상 지정 후 확인 필요
8,재현 기록,requirements 또는 lock 파일에 기록하는가?,REVIEW,설치 대상 지정 후 확인 필요
9,정책,수업 또는 조직의 설치 정책에 맞는가?,REVIEW,설치 대상 지정 후 확인 필요



새 패키지 설치 결정: DO_NOT_INSTALL_UNTIL_REVIEWED
이번 셀에서는 패키지를 설치하지 않았습니다.
Sandbox 준비 여부와 코드 실행 승인 여부는 별도로 확인해야 합니다.


### STEP 6. Sandbox와 Package 검토 — 결과 확인

#### 결과 관찰

현재 노트북에서 사용하는 Python과 패키지 버전을 확인했다.

| 항목 | 현재 버전 |
|---|---|
| Python | 3.14.6 |
| pandas | 3.0.5 |
| numpy | 2.5.1 |

Sandbox 검토표 9개 항목과 새 패키지 설치 검토표 10개 항목을 작성했다. 실제 설정이나 설치 대상의 Evidence를 확인하지 않았으므로, 총 19개 항목 모두 REVIEW로 표시됐다.

새 패키지 설치 결정은 **DO_NOT_INSTALL_UNTIL_REVIEWED**이며, 이번 단계에서 패키지를 설치하지 않았다.

#### 나의 해석과 판단

검토표를 만들었다는 사실과 실행 환경이 안전하게 준비됐다는 사실은 다르다.

입력 보호, 인증정보 분리, 출력 경로 제한, 네트워크 차단, 자원 제한, 실행 전후 파일 비교 조건은 실제 설정과 기록으로 확인해야 한다. 현재는 이를 입증하지 않았으므로 REVIEW 상태를 유지한다.

패키지 버전을 조회한 것만으로 Python 호환성이나 설치 안전성까지 확인됐다고 판단하지 않는다.

#### Evidence

현재 실행 환경의 버전 출력, Sandbox 검토표, Package 검토표, 설치 결정 메시지를 확인했다.

검토표의 Evidence 칸은 추가 확인이 필요하다는 안내이며, 실제 설정을 입증하는 자료는 아직 아니다.

#### 수정 내용

이번 단계에서는 현재 버전을 조회하고 검토표를 작성했다. Sandbox 설정 변경이나 새 패키지 설치는 수행하지 않았다.

#### 한계와 추가 확인 사항

가상환경 `.venv`는 패키지 환경을 분리하지만, 파일 접근 제한·네트워크 차단·자원 제한까지 자동으로 제공하지 않는다.

이후 제한 실행 전에는 실행 대상 코드에 필요한 Sandbox 조건을 준비하고, 각 조건의 Evidence를 기록해야 한다. 새 패키지 설치가 필요하면 대상 패키지의 이름·출처·버전·의존성을 구체적으로 검토해야 한다.

현재 결과는 **검토 항목 정리 완료, 실행 환경 준비 여부는 추가 확인 필요**로 판단한다.

### STEP 7. 전체 Evidence와 Execution Gate 확인

In [21]:
import pandas as pd
import numpy as np


# 1. STEP 2: 검증용 복사본의 스키마·키 검증 결과
schema_passed = check_result["문제 건수"].eq(0).all()
schema_status = "PASS" if schema_passed else "FAIL"


# 2. STEP 3: 집계 검사와 총합 일치 여부
aggregate_passed = (
    validation_result["문제 건수"].eq(0).all()
    and np.isclose(
        source_total, category_total, rtol=0, atol=0.01
    )
    and np.isclose(
        source_total, monthly_total, rtol=0, atol=0.01
    )
)

aggregate_status = "PASS" if aggregate_passed else "FAIL"


# 3. STEP 5: 회귀·분류 Feature 검토 결과
feature_audit = pd.concat(
    [regression_audit, classification_audit],
    ignore_index=True,
)

# 현재 출력에서 확인한 REVIEW 항목 수
feature_review_count = feature_audit["status"].eq("REVIEW").sum()

# 예측 시점 가용성의 Evidence는 아직 확인 전
ml_status = "REVIEW"


# 4. STEP 4: 정적 스캔의 차단 항목 확인
severity = static_scan["severity"].str.lower()
blocked_findings = severity.isin(["critical", "high"]).sum()

# 탐지 0건이어도 안전을 보장하지 않으므로 REVIEW
static_status = "BLOCKED" if blocked_findings > 0 else "REVIEW"


# 5. STEP 6: 아직 확인되지 않은 환경·설치 조건
sandbox_review_count = sandbox_checklist["상태"].eq("REVIEW").sum()
package_review_count = package_review["상태"].eq("REVIEW").sum()

sandbox_package_status = "REVIEW"


# 6. 사람의 실행 승인은 아직 없음
execution_approved = False
human_status = "REVIEW"


# 7. 각 검증 축을 하나의 표로 정리
execution_gate = pd.DataFrame([
    (
        "schema_and_keys",
        schema_status,
        "STEP 2: FK 문제 행을 제외한 검증용 복사본 재검증",
    ),
    (
        "aggregate_validation",
        aggregate_status,
        "STEP 3: 날짜가 있는 completed 주문의 동일 범위 총합 대조",
    ),
    (
        "ml_leakage",
        ml_status,
        f"STEP 5: Feature REVIEW {feature_review_count}건, 시점 확인 필요",
    ),
    (
        "static_scan",
        static_status,
        f"STEP 4: 공식 위험 예제의 critical/high 탐지 {blocked_findings}건",
    ),
    (
        "sandbox_and_package",
        sandbox_package_status,
        f"STEP 6: Sandbox REVIEW {sandbox_review_count}건, "
        f"Package REVIEW {package_review_count}건",
    ),
    (
        "human_approval",
        human_status,
        f"execution_approved = {execution_approved}",
    ),
], columns=["검증 축", "상태", "Evidence"])


# FAIL 또는 BLOCKED가 하나라도 있으면 실행 금지
has_blocker = execution_gate["상태"].isin(["FAIL", "BLOCKED"]).any()

if has_blocker:
    execution_decision = "DO_NOT_EXECUTE"
else:
    execution_decision = "HUMAN_REVIEW_REQUIRED"

print("Execution Gate")
display(execution_gate)

print(f"\n최종 실행 결정: {execution_decision}")
print("검증 대상 코드는 실행하지 않았습니다.")

print("\n집계 PASS의 범위:")
print("FK 문제 행과 날짜 결측 주문을 제외한 completed 주문 상세")
print("2027-01 날짜의 업무상 타당성은 추가 확인이 필요합니다.")

Execution Gate


,검증 축,상태,Evidence
0,schema_and_keys,PASS,STEP 2: FK 문제 행을 제외한 검증용 복사본 재검증
1,aggregate_validation,PASS,STEP 3: 날짜가 있는 completed 주문의 동일 범위 총합 대조
2,ml_leakage,REVIEW,"STEP 5: Feature REVIEW 12건, 시점 확인 필요"
3,static_scan,BLOCKED,STEP 4: 공식 위험 예제의 critical/high 탐지 3건
4,sandbox_and_package,REVIEW,"STEP 6: Sandbox REVIEW 9건, Package REVIEW 10건"
5,human_approval,REVIEW,execution_approved = False



최종 실행 결정: DO_NOT_EXECUTE
검증 대상 코드는 실행하지 않았습니다.

집계 PASS의 범위:
FK 문제 행과 날짜 결측 주문을 제외한 completed 주문 상세
2027-01 날짜의 업무상 타당성은 추가 확인이 필요합니다.


### STEP 7. Evidence와 Execution Gate 확인 — 결과 확인

#### 결과 관찰

노트북에서 확인한 검증 결과를 모아 Execution Gate를 작성했다.

| 검증 축 | 상태 | 주요 근거 |
|---|---|---|
| schema_and_keys | PASS | FK 문제 행을 제외한 복사본의 성공 기준 6가지 통과 |
| aggregate_validation | PASS | 날짜가 있는 completed 주문 범위에서 기준·카테고리·월별 총합 일치 |
| ml_leakage | REVIEW | Feature 12개의 예측 시점 가용성 확인 필요 |
| static_scan | BLOCKED | 공식 위험 예제에서 high 항목 3건 탐지 |
| sandbox_and_package | REVIEW | Sandbox 9개, Package 10개 항목의 Evidence 확인 필요 |
| human_approval | REVIEW | execution_approved = False |

최종 실행 결정은 **DO_NOT_EXECUTE**였다. 정적 스캔 대상 코드는 실행하지 않았다.

#### 나의 해석과 판단

FAIL 또는 BLOCKED가 하나라도 있으면 실행하지 않는 원칙을 적용했다. 이번에는 static_scan이 BLOCKED이므로 실행 금지로 판단됐다.

DO_NOT_EXECUTE는 검증 과정이 실패했다는 뜻이 아니라, 현재 검사한 코드에 실행 차단 사유가 있다는 뜻이다.

데이터의 스키마·키와 집계 검증이 통과해도 외부 요청·파일 쓰기의 안전성, 실행 환경, 사람 승인은 별도로 확인해야 한다. 차단 항목이 없어지더라도 자동으로 실행 승인 상태가 되지는 않는다.

#### Evidence

Execution Gate 결과표, STEP 4의 high 탐지 3건, STEP 5·6의 REVIEW 결과, execution_approved = False를 확인했다.

STEP 3의 기준·카테고리·월별 총합은 모두 138,985,000이었다.

#### 수정 내용

기존 검증 결과를 노트북의 Execution Gate 표로 정리했다. 검증 대상 코드를 수정하거나 실행하지 않았으며, 사람의 실행 승인 상태도 False로 유지했다.

#### 한계와 추가 확인 사항

데이터 검증의 PASS는 FK 문제 행과 날짜 결측 주문을 제외한 범위에 해당한다. 또한 월별 결과에 포함된 2027-01 날짜의 업무상 타당성은 추가 확인이 필요하다.

정적 스캔의 BLOCKED는 현재 검사한 공식 위험 예제에 대한 결과다. 이후 실행 후보 코드를 수정하거나 바꾸면 해당 코드로 다시 검사해야 한다.

이번 Evidence는 노트북에서 정리한 표이며, 공식 일괄 보고서 생성 스크립트를 실행한 결과는 아니다. 제한 실행과 실행 후 검증도 아직 수행하지 않았다.

### STEP 8. 사람의 APPROVE / REVISE / BLOCK 판단

In [22]:
import pandas as pd
from datetime import datetime


# 1. 현재 검사한 코드에 대한 판단 초안
review_target = "DEFAULT_STATIC_SCAN_EXAMPLE"
decision_draft = "BLOCK"

# 실행 승인은 아직 없음
execution_approved = False


# 2. 분석 타당성과 실행 안전을 함께 검토
human_review = pd.DataFrame([
    (
        "분석 타당성",
        "추가 검증 필요",
        "외부에서 받을 파일의 스키마·키·값 검증이 없음. "
        "STEP 2·3에서 검사한 로컬 데이터와 다른 입력임.",
    ),
    (
        "실행 안전",
        "차단 사유 있음",
        f"정적 스캔에서 critical/high {blocked_findings}건 탐지. "
        "외부 요청과 파일 덮어쓰기의 목적·범위 검토 필요.",
    ),
    (
        "실행 환경",
        "추가 검증 필요",
        "Sandbox 설정의 Evidence 확인이 완료되지 않음.",
    ),
], columns=["검토 축", "판단 근거 상태", "근거"])

print("사람 검토용 판단 근거")
display(human_review)


# 3. 수정이 필요한 내용 정리
revision_plan = pd.DataFrame([
    (
        "입력",
        "검증한 로컬 입력을 사용하는 코드로 수정하거나, "
        "외부 입력의 출처·구조·값을 검증",
    ),
    (
        "외부 요청",
        "분석에 필요한지 확인하고 불필요한 요청 제거",
    ),
    (
        "파일 저장",
        "승인된 출력 폴더와 기존 파일 덮어쓰기 방지 조건 적용",
    ),
    (
        "재검토",
        "수정한 코드로 정적 스캔과 Execution Gate 다시 확인",
    ),
], columns=["수정 대상", "수정 계획"])

print("\n수정 계획 — 아직 적용하지 않음")
display(revision_plan)


# 4. 현재까지의 검토·수정 상태 기록
human_revision_log = pd.DataFrame([{
    "recorded_at": datetime.now().isoformat(timespec="seconds"),
    "review_target": review_target,
    "decision_draft": decision_draft,
    "human_review_status": "판단 초안 — 사람이 근거 확인 필요",
    "revision_status": "NOT_REVISED",
    "execution_approved": execution_approved,
    "limited_execution": "NOT_RUN",
}])

print("\n사람 검토·수정 기록")
display(human_revision_log)

print(f"\n판단 초안: {decision_draft}")
print(f"실행 승인: {execution_approved}")
print("검증 대상 코드의 수정이나 실행은 수행하지 않았습니다.")

사람 검토용 판단 근거


,검토 축,판단 근거 상태,근거
0,분석 타당성,추가 검증 필요,외부에서 받을 파일의 스키마·키·값 검증이 없음. STEP 2·3에서 검사한 로컬 ...
1,실행 안전,차단 사유 있음,정적 스캔에서 critical/high 3건 탐지. 외부 요청과 파일 덮어쓰기의 목...
2,실행 환경,추가 검증 필요,Sandbox 설정의 Evidence 확인이 완료되지 않음.



수정 계획 — 아직 적용하지 않음


,수정 대상,수정 계획
0,입력,"검증한 로컬 입력을 사용하는 코드로 수정하거나, 외부 입력의 출처·구조·값을 검증"
1,외부 요청,분석에 필요한지 확인하고 불필요한 요청 제거
2,파일 저장,승인된 출력 폴더와 기존 파일 덮어쓰기 방지 조건 적용
3,재검토,수정한 코드로 정적 스캔과 Execution Gate 다시 확인



사람 검토·수정 기록


,recorded_at,review_target,decision_draft,human_review_status,revision_status,execution_approved,limited_execution
0,2026-10-06T10:54:58,DEFAULT_STATIC_SCAN_EXAMPLE,BLOCK,판단 초안 — 사람이 근거 확인 필요,NOT_REVISED,False,NOT_RUN



판단 초안: BLOCK
실행 승인: False
검증 대상 코드의 수정이나 실행은 수행하지 않았습니다.


### STEP 8. 사람의 실행 판단과 수정 기록 — 결과 확인

#### 결과 관찰

공식 위험 예제에 대해 분석 타당성, 실행 안전, 실행 환경을 검토할 근거를 정리했다.

| 검토 축 | 확인한 상태 |
|---|---|
| 분석 타당성 | 외부 입력의 스키마·키·값 검증 필요 |
| 실행 안전 | critical/high 탐지 3건으로 차단 사유 존재 |
| 실행 환경 | Sandbox 설정의 Evidence 확인 필요 |

현재 기록된 상태는 다음과 같다.

| 항목 | 상태 |
|---|---|
| 검토 대상 | DEFAULT_STATIC_SCAN_EXAMPLE |
| 판단 초안 | BLOCK |
| 사람 검토 상태 | 근거 확인 필요 |
| 코드 수정 | NOT_REVISED |
| 실행 승인 | False |
| 제한 실행 | NOT_RUN |

#### 나의 해석과 판단

현재 예제는 외부 요청과 파일 쓰기를 포함하며, 외부 입력의 분석 적합성과 실행 환경도 확인되지 않았다. 따라서 현재 상태의 실행을 보류하는 BLOCK 판단 초안을 작성했다.

이 기록은 검토 근거를 정리한 초안이다. 사람의 최종 검토 완료나 실행 승인을 기록한 것은 아니다.

#### Evidence

사람 검토용 판단 근거표, 수정 계획표, human_revision_log를 확인했다. 실행 승인 값은 False이며, 검증 대상 코드는 수정하거나 실행하지 않았다.

#### 수정 내용

입력 검증, 불필요한 외부 요청 제거, 출력 경로 제한, 덮어쓰기 방지, 수정 후 재검토를 계획했다.

현재는 수정 계획만 작성했으며, 실제 코드 변경은 수행하지 않았다.

#### 한계와 추가 확인 사항

사람이 근거를 직접 확인하고 최종 판단을 기록해야 한다. 수정한 코드가 생기면 해당 코드에 대해 분석 검증과 정적 스캔, 실행 환경 검토를 다시 수행해야 한다.

현재 승인 상태에서는 STEP 9의 제한 실행을 진행하지 않는다. 제한 실행을 하지 않았으므로 실행 후 검증도 아직 수행하지 않았다.

### STEP 9. 승인된 코드만 제한 환경에서 실행

In [23]:
import pandas as pd
from datetime import datetime

# Execution Gate의 차단 사유 확인
blockers = execution_gate.loc[
    execution_gate["상태"].isin(["FAIL", "BLOCKED"]),
    "검증 축",
].tolist()

# 현재 승인과 차단 상태 확인
print(f"판단 초안: {decision_draft}")
print(f"실행 승인: {execution_approved}")
print(f"Execution Gate: {execution_decision}")
print(f"차단 항목: {blockers}")

# 승인된 실행 대상과 환경이 준비된 경우에는 별도 실행 코드가 필요함
if execution_approved and not blockers:
    raise RuntimeError(
        "승인한 코드 버전과 제한 실행 환경을 확인한 뒤 "
        "해당 코드의 실행 절차를 준비해야 합니다."
    )

# 현재는 실행하지 않고 보류 이유만 기록
limited_run_log = pd.DataFrame([{
    "recorded_at": datetime.now().isoformat(timespec="seconds"),
    "target": review_target,
    "execution_approved": execution_approved,
    "execution_decision": execution_decision,
    "run_status": "NOT_RUN",
    "reason": "실행 승인 없음 또는 Execution Gate 차단 사유 존재",
    "start_time": None,
    "end_time": None,
    "exit_code": None,
    "post_execution_evidence": "실행하지 않아 수집하지 않음",
}])

print("\nSTEP 9 제한 실행 기록")
display(limited_run_log)

print("\n제한 실행 보류 — 검증 대상 코드는 실행하지 않았습니다.")

판단 초안: BLOCK
실행 승인: False
Execution Gate: DO_NOT_EXECUTE
차단 항목: ['static_scan']

STEP 9 제한 실행 기록


,recorded_at,target,execution_approved,execution_decision,run_status,reason,start_time,end_time,exit_code,post_execution_evidence
0,2026-10-06T11:00:08,DEFAULT_STATIC_SCAN_EXAMPLE,False,DO_NOT_EXECUTE,NOT_RUN,실행 승인 없음 또는 Execution Gate 차단 사유 존재,None,None,None,실행하지 않아 수집하지 않음



제한 실행 보류 — 검증 대상 코드는 실행하지 않았습니다.


### STEP 9. 승인된 코드의 제한 실행 — 결과 확인

#### 결과 관찰

실행 전 승인 상태와 Execution Gate를 확인했다.

| 항목 | 결과 |
|---|---|
| 검증 대상 | DEFAULT_STATIC_SCAN_EXAMPLE |
| 판단 초안 | BLOCK |
| 실행 승인 | False |
| Execution Gate | DO_NOT_EXECUTE |
| 차단 항목 | static_scan |
| 제한 실행 상태 | NOT_RUN |

실행 승인과 실행 조건이 충족되지 않아 검증 대상 코드를 실행하지 않았다.

#### 나의 해석과 판단

STEP 9는 APPROVE된 코드에만 적용하는 단계다. 현재는 정적 스캔의 차단 사유가 남아 있고 사람의 실행 승인도 없으므로, 제한 실행을 보류했다.

NOT_RUN은 코드 실행 실패가 아니라 실행 조건을 충족하지 않아 실행하지 않았다는 뜻이다.

#### Evidence

승인 상태 출력, Execution Gate의 차단 항목, limited_run_log를 확인했다. 로그에는 실행 보류 사유와 NOT_RUN 상태가 기록됐다.

#### 수행 내용

대상 코드를 실행하지 않고 승인·차단 상태를 점검하여 보류 기록만 작성했다.

실제 실행이 없으므로 시작 시각, 종료 시각, 종료 코드는 None으로 유지했다.

#### 한계와 추가 확인 사항

제한 실행을 수행하지 않았으므로 실제 실행 동작과 실행 후 변경 사항은 확인하지 않았다. 사후 검증 Evidence도 수집하지 않았다.

코드를 수정한 뒤 재검토하고, 실행 환경을 준비하여 사람이 승인한 경우에만 제한 실행을 진행할 수 있다.

### STEP 10. 실행 성공 메시지보다 Post-execution Validation 확인

In [24]:
import pandas as pd

# STEP 9에서 실행하지 않았는지 확인
was_not_run = limited_run_log["run_status"].eq("NOT_RUN").all()

assert was_not_run, (
    "실행 이력이 있습니다. 실제 실행 결과를 대상으로 사후 검증해야 합니다."
)

# 실행 후 확인해야 할 항목 정리
post_validation = pd.DataFrame([
    ("입력 행 수", "실행 전후 입력 행 수 비교"),
    ("출력 행 수", "실제 출력과 예상 행 수 비교"),
    ("merge 행 수", "merge 전후 행 수 비교"),
    ("미매칭", "연결되지 않은 행 수 확인"),
    ("completed 범위", "집계에 포함된 주문 상태 확인"),
    ("금액 계산식", "line_total = quantity × unit_price 확인"),
    ("집계 총합", "기준·카테고리·월별 총합 대조"),
    ("생성 파일", "예상한 파일만 생성됐는지 확인"),
    ("출력 경로", "허용 경로 밖의 변경 여부 확인"),
    ("원본 보존", "입력 파일의 해시·크기 등을 비교"),
    ("외부 전송", "외부 네트워크 전송 여부 확인"),
    ("민감정보 로그", "로그에 개인정보·인증정보가 남았는지 확인"),
], columns=["검증 항목", "확인할 내용"])

# 실행이 없으므로 사후 검증 결과를 PASS로 표시하지 않음
post_validation["상태"] = "NOT_EVALUATED"
post_validation["사유"] = "STEP 9에서 검증 대상 코드를 실행하지 않음"

print("STEP 10 실행 후 검증 기록")
display(post_validation)

post_validation_status = "NOT_EVALUATED"

print(f"\n사후 검증 상태: {post_validation_status}")
print("STEP 2·3의 결과는 제한 실행 후 검증 결과로 대체해 기록하지 않습니다.")

STEP 10 실행 후 검증 기록


,검증 항목,확인할 내용,상태,사유
0,입력 행 수,실행 전후 입력 행 수 비교,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
1,출력 행 수,실제 출력과 예상 행 수 비교,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
2,merge 행 수,merge 전후 행 수 비교,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
3,미매칭,연결되지 않은 행 수 확인,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
4,completed 범위,집계에 포함된 주문 상태 확인,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
5,금액 계산식,line_total = quantity × unit_price 확인,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
6,집계 총합,기준·카테고리·월별 총합 대조,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
7,생성 파일,예상한 파일만 생성됐는지 확인,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
8,출력 경로,허용 경로 밖의 변경 여부 확인,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음
9,원본 보존,입력 파일의 해시·크기 등을 비교,NOT_EVALUATED,STEP 9에서 검증 대상 코드를 실행하지 않음



사후 검증 상태: NOT_EVALUATED
STEP 2·3의 결과는 제한 실행 후 검증 결과로 대체해 기록하지 않습니다.


### STEP 10. 실행 후 검증 — 결과 확인

#### 결과 관찰

STEP 9에서 검증 대상 코드의 제한 실행을 수행하지 않았으므로, 실행 후 검증 항목 12개를 모두 NOT_EVALUATED로 기록했다.

검증 항목은 입력·출력 행 수, merge 행 수, 미매칭, completed 범위, 금액 계산식, 집계 총합, 생성 파일, 출력 경로, 원본 보존, 외부 전송, 민감정보 로그다.

#### 나의 해석과 판단

NOT_EVALUATED는 검증 통과나 실패가 아니라, 실제 실행 결과가 없어 평가하지 않았다는 뜻이다.

STEP 2·3에서 수행한 데이터 검증은 제한 실행 후의 사후 검증과 다르다. 앞선 PASS 결과를 이번 단계의 PASS로 대신 기록하지 않았다.

#### Evidence

post_validation 결과표에서 12개 항목 모두 NOT_EVALUATED이며, 사유는 “STEP 9에서 검증 대상 코드를 실행하지 않음”으로 표시됐다.

STEP 9의 limited_run_log에도 NOT_RUN이 기록되어 있다.

#### 수행 내용

실행 후 확인해야 할 항목을 정리하고 미평가 사유를 기록했다. 실제 출력 파일 검사, 원본 해시 비교, 네트워크·로그 검증은 수행하지 않았다.

#### 한계와 추가 확인 사항

현재 결과로는 대상 코드의 실행 동작, 출력 정확성, 파일 변경, 외부 전송 여부를 판단할 수 없다.

승인된 코드를 제한 실행한 경우에는 실제 실행 전후 Evidence를 수집하여 사후 검증을 수행해야 한다. 현재는 **실행 후 검증 미수행** 상태를 유지한다.

### STEP 11. 오류를 LLM에게 다시 물을 때도 최소 Context만 공유

In [25]:
from src.llm_code_validation import build_error_fix_prompt_template


# 1. 공식 오류 수정 요청 템플릿 확인
error_fix_template = build_error_fix_prompt_template()

print("공식 오류 수정 요청 템플릿")
print(error_fix_template)


# 2. 앞서 발생한 날짜 오류를 최소 정보로 정리
# 고객 원본 행, 인증정보, 개인 경로는 포함하지 않음
error_fix_example = """
[분석 목적]
completed 주문의 카테고리별·월별 금액을 집계하고,
같은 범위에서 기준 총합과 집계 총합을 비교하려고 합니다.

[필요한 데이터 구조]
orders: order_id, order_date, order_status
order_items: order_id, product_id, line_total
products: product_id, category

[오류 당시의 Evidence]
FK 문제 행 제외 후 orders는 298행, order_items는 745행입니다.
주문 날짜 결측·변환 실패는 3건이며, 이 중 completed는 2건입니다.
날짜 제외 전 completed 주문 상세 금액은 140,888,000입니다.

[문제를 확인하는 최소 코드]
parsed_dates = pd.to_datetime(orders["order_date"], errors="coerce")
invalid_dates = parsed_dates.isna()
assert not invalid_dates.any(), "날짜 문제가 발견됐습니다."

[오류 메시지]
AssertionError: 날짜 문제가 발견됐습니다.

[요청]
1. 이 오류의 원인을 설명해 주세요.
2. 날짜를 임의로 채우지 않는 처리 방안을 제안해 주세요.
3. 원본을 보존하고 제외 행 수와 금액을 기록해 주세요.
4. 카테고리·월별 집계 범위를 일치시키는 방법을 제안해 주세요.
5. 실행이나 새 패키지 설치 없이 코드와 검증 기준만 제안해 주세요.
"""

print("\n최소 Context 작성 예시 — 앞서 발생한 오류")
print(error_fix_example)

print("\n실제 외부 LLM 호출은 수행하지 않았습니다.")
print("실제 공유 전에는 코드 문자열과 URL에도 민감정보가 없는지 확인하세요.")

공식 오류 수정 요청 템플릿
# 오류 수정 요청 — 최소·비식별 Context만 사용

분석 목적:
- [계산/예측하려는 내용을 한 문장으로 작성]

실제 필요한 데이터 구조:
- [데이터셋과 필요한 컬럼만 작성]
- 원본 고객/거래 행, 개인정보, API Key, 내부 URL, 사용자명, 절대 경로는 제공하지 않음

최소 재현 코드:
```python
[오류를 재현하는 최소 코드만]
```

민감정보를 제거한 오류 또는 검증 결과:
```text
[예외 유형, 필요한 메시지, 행 수, 미매칭 수, 총합 차이 등]
```

요청:
1. 실행 오류와 분석 논리 오류를 구분해서 설명해 주세요.
2. 실제 제공한 컬럼명만 사용하는 최소 수정안을 제안해 주세요.
3. merge validate, 전후 행 수, 미매칭, 총합 대조를 포함해 주세요.
4. 원인을 확인하지 못한 부분은 추측하지 말고 확인 방법을 제시해 주세요.
5. 파일 삭제·덮어쓰기, 외부 통신, OS 명령, package 설치 코드를 추가하지 마세요.
6. 수정 코드와 함께 사람이 다시 검증할 항목을 표로 제시해 주세요.


최소 Context 작성 예시 — 앞서 발생한 오류

[분석 목적]
completed 주문의 카테고리별·월별 금액을 집계하고,
같은 범위에서 기준 총합과 집계 총합을 비교하려고 합니다.

[필요한 데이터 구조]
orders: order_id, order_date, order_status
order_items: order_id, product_id, line_total
products: product_id, category

[오류 당시의 Evidence]
FK 문제 행 제외 후 orders는 298행, order_items는 745행입니다.
주문 날짜 결측·변환 실패는 3건이며, 이 중 completed는 2건입니다.
날짜 제외 전 completed 주문 상세 금액은 140,888,000입니다.

[문제를 확인하는 최소 코드]
parsed_dates = pd.to_datetime(

In [26]:
# 공식 템플릿을 제외하고 작성한 예시만 출력
print(error_fix_example)


[분석 목적]
completed 주문의 카테고리별·월별 금액을 집계하고,
같은 범위에서 기준 총합과 집계 총합을 비교하려고 합니다.

[필요한 데이터 구조]
orders: order_id, order_date, order_status
order_items: order_id, product_id, line_total
products: product_id, category

[오류 당시의 Evidence]
FK 문제 행 제외 후 orders는 298행, order_items는 745행입니다.
주문 날짜 결측·변환 실패는 3건이며, 이 중 completed는 2건입니다.
날짜 제외 전 completed 주문 상세 금액은 140,888,000입니다.

[문제를 확인하는 최소 코드]
parsed_dates = pd.to_datetime(orders["order_date"], errors="coerce")
invalid_dates = parsed_dates.isna()
assert not invalid_dates.any(), "날짜 문제가 발견됐습니다."

[오류 메시지]
AssertionError: 날짜 문제가 발견됐습니다.

[요청]
1. 이 오류의 원인을 설명해 주세요.
2. 날짜를 임의로 채우지 않는 처리 방안을 제안해 주세요.
3. 원본을 보존하고 제외 행 수와 금액을 기록해 주세요.
4. 카테고리·월별 집계 범위를 일치시키는 방법을 제안해 주세요.
5. 실행이나 새 패키지 설치 없이 코드와 검증 기준만 제안해 주세요.



### STEP 11. 오류 수정 요청의 최소 Context 작성 — 결과 확인

#### 결과 관찰

공식 오류 수정 요청 템플릿을 확인하고, STEP 3에서 발생했던 날짜 검증 오류를 최소 Context 예시로 작성했다.

분석 목적, 필요한 데이터셋·컬럼, 오류를 확인하는 핵심 코드, 오류 메시지, 행 수·날짜 결측 수·집계 금액을 포함했다.

실제 외부 LLM 호출은 수행하지 않았다.

#### 나의 해석과 판단

오류를 질문할 때 전체 데이터나 전체 로그를 전달할 필요는 없다. 원인 파악에 필요한 구조와 검증 수치를 중심으로 공유해야 한다.

이번 예시는 날짜 문제를 처리하기 전의 오류 상황을 설명한 것이다. 날짜가 있는 주문으로 범위를 제한하여 검증을 통과한 현재 결과와 구분한다.

#### Evidence

공식 템플릿 출력, error_fix_example 문자열, 외부 LLM 호출을 수행하지 않았다는 출력 메시지를 확인했다.

출력 일부가 길이 제한으로 접혔지만 실행 오류는 발생하지 않았다.

#### 수행 내용

고객 원본 행, 인증정보, 내부 URL, 사용자명, 개인 절대 경로를 포함하지 않는 예시를 작성했다.

날짜 임의 보정 대신 원본 보존, 제외 행 수·금액 기록, 집계 범위 일치를 요청하도록 구성했다.

#### 한계와 추가 확인 사항

요청 예시를 작성한 것이며, 외부 LLM의 답변을 받거나 수정안을 검증한 단계는 아니다.

실제 공유 전에는 오류 메시지뿐 아니라 코드 안의 문자열과 URL에도 민감정보가 없는지 다시 확인해야 한다. 이후 제안받은 코드는 실행 전 검토와 검증을 거쳐야 한다.

### STEP 12. Evidence 파일과 최종 신뢰 범위 확인

In [27]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from hashlib import sha256


# 1. 집계 총합을 다시 확인
assert np.isclose(
    source_total, category_total, rtol=0, atol=0.01
), "카테고리 총합 불일치"

assert np.isclose(
    source_total, monthly_total, rtol=0, atol=0.01
), "월별 총합 불일치"


# 2. 실제 입력 파일의 이름·크기·해시 기록
# 해시는 나중에 같은 입력 파일인지 비교하는 데 사용
input_names = [
    "customers_clean.csv",
    "products_cleaned.csv",
    "orders_clean.csv",
    "order_items_clean.csv",
]

input_records = []

for name in input_names:
    file_path = data_dir / name
    assert file_path.is_file(), f"입력 파일 없음: {name}"

    file_bytes = file_path.read_bytes()

    input_records.append({
        "파일명": name,
        "크기_bytes": len(file_bytes),
        "SHA256": sha256(file_bytes).hexdigest(),
    })

input_inventory = pd.DataFrame(input_records)


# 3. 원본·STEP 2·STEP 3의 데이터 범위 기록
step3_data = {
    "customers": validated_datasets["customers"],
    "products": products_check,
    "orders": orders_check,
    "order_items": items_check,
}

dataset_records = []

for name in datasets:
    dataset_records.append({
        "데이터": name,
        "최초_행수": len(datasets[name]),
        "STEP2_행수": len(validated_datasets[name]),
        "STEP3_행수": len(step3_data[name]),
        "컬럼": ", ".join(datasets[name].columns),
    })

dataset_inventory = pd.DataFrame(dataset_records)


# 4. 기존 파일을 덮어쓰지 않는 새 결과 폴더 생성
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
report_dir = Path.cwd() / "reports" / f"ch12_{run_id}"
report_dir.mkdir(parents=True, exist_ok=False)


# 5. 실제로 확인한 결과표 저장
report_tables = {
    "ch12_input_inventory.csv": input_inventory,
    "ch12_dataset_inventory.csv": dataset_inventory,
    "ch12_required_column_check.csv": check_result[
        check_result["성공 기준"].str.startswith("①")
    ],
    "ch12_primary_key_check.csv": check_result[
        check_result["성공 기준"].str.startswith(("②", "③"))
    ],
    "ch12_relationship_key_check.csv": check_result[
        check_result["성공 기준"].str.startswith(("④", "⑤", "⑥"))
    ],
    "ch12_category_sales_validated.csv": category_sales,
    "ch12_monthly_sales_validated.csv": monthly_sales,
    "ch12_aggregate_validation.csv": validation_result,
    "ch12_aggregate_totals.csv": total_check,
    "ch12_scope_change.csv": scope_change,
    "ch12_ml_leakage_review.csv": feature_audit,
    "ch12_generated_code_static_scan.csv": static_scan,
    "ch12_execution_gate.csv": execution_gate,
    "ch12_sandbox_execution_checklist.csv": sandbox_checklist,
    "ch12_package_install_review.csv": package_review,
    "ch12_human_review_draft.csv": human_review,
    "ch12_human_revision_log.csv": human_revision_log,
    "ch12_revision_plan.csv": revision_plan,
    "ch12_limited_run_log.csv": limited_run_log,
    "ch12_post_validation.csv": post_validation,
    "ch12_environment.csv": environment_info,
}

# utf-8-sig는 Excel에서 한글을 읽기 편한 인코딩
for filename, table in report_tables.items():
    table.to_csv(
        report_dir / filename,
        index=False,
        encoding="utf-8-sig",
    )


# 6. 검사한 코드와 오류 요청 텍스트 저장
# 코드는 실행 파일이 아닌 텍스트 파일로 보관
(report_dir / "ch12_scanned_code.txt").write_text(
    generated_code,
    encoding="utf-8",
)

(report_dir / "ch12_error_fix_prompt_template.md").write_text(
    error_fix_template,
    encoding="utf-8",
)

(report_dir / "ch12_error_fix_example.md").write_text(
    error_fix_example,
    encoding="utf-8",
)


# 7. 최종 신뢰 범위와 한계 기록
final_trust = "사용 보류"
code_hash = sha256(generated_code.encode("utf-8")).hexdigest()

summary = f"""# Chapter 12 검증 결과 요약

## Evidence 생성 방식
노트북에서 확인한 결과표를 직접 저장했다.
공식 일괄 보고서 생성 스크립트를 실행한 결과는 아니다.

## 검증 대상 코드
- 대상: {review_target}
- 코드 SHA256: {code_hash}
- 판단 초안: {decision_draft}
- 실행 승인: {execution_approved}
- Execution Gate: {execution_decision}
- 제한 실행: NOT_RUN
- 사후 검증: {post_validation_status}

## 확인한 데이터 범위
- 최초 FK 미매칭: 관계 3곳에서 각각 2건
- STEP 2: 주문 2행, 주문 상세 7행 제외
- STEP 3: 날짜 결측 주문 3행과 연결된 주문 상세 9행 제외
- 집계 대상: 날짜가 있는 completed 주문
- completed 주문: {len(completed_ids)}건
- completed 주문 상세: {len(completed)}행

## 총합 대조
- 기준 총합: {source_total:,.2f}
- 카테고리 총합: {category_total:,.2f}
- 월별 총합: {monthly_total:,.2f}
- 날짜 결측으로 제외한 completed 금액: {before_total - source_total:,.2f}

## 신뢰할 수 있는 범위
검사한 복사본의 필수 컬럼과 PK/FK 관계를 확인했다.
날짜가 있는 completed 주문 범위에서 금액 계산,
merge 행 수·미매칭, 카테고리·월별 총합 일치를 확인했다.

## 추가 확인이 필요한 범위
- 제외한 행이 분석 결과에 미치는 영향
- 2027-01 날짜의 업무상 타당성
- Feature의 예측 시점 가용성
- Sandbox 설정과 패키지 설치 조건
- 사람의 최종 검토와 승인
- 승인된 코드의 제한 실행 및 사후 검증

## 최종 신뢰 판단
{final_trust}

이 판단은 현재 정적 스캔 대상인 공식 위험 예제에 대한 것이다.
데이터 검증의 PASS를 코드 실행 안전성의 PASS로 해석하지 않는다.
집계 금액은 완료 주문 상세 금액이며 회계상 순매출로 단정하지 않는다.
"""

(report_dir / "ch12_code_validation_summary.md").write_text(
    summary,
    encoding="utf-8",
)


# 8. 저장된 파일의 존재 여부와 크기 확인
saved_records = []

for file_path in sorted(report_dir.iterdir()):
    saved_records.append({
        "파일명": file_path.name,
        "존재": file_path.is_file(),
        "크기_bytes": file_path.stat().st_size,
    })

saved_files = pd.DataFrame(saved_records)

print("저장된 Evidence 파일")
display(saved_files)

assert saved_files["존재"].all(), "저장되지 않은 파일이 있습니다."
assert saved_files["크기_bytes"].gt(0).all(), "빈 파일이 있습니다."

print(f"\n저장 위치: {report_dir}")
print(f"저장 파일 수: {len(saved_files)}개")
print(f"현재 위험 예제의 최종 신뢰 판단: {final_trust}")
print("Evidence 저장 완료 — 제한 실행과 사후 검증은 미수행 상태입니다.")

저장된 Evidence 파일


,파일명,존재,크기_bytes
0,ch12_aggregate_totals.csv,True,105
1,ch12_aggregate_validation.csv,True,269
2,ch12_category_sales_validated.csv,True,172
3,ch12_code_validation_summary.md,True,1886
4,ch12_dataset_inventory.csv,True,415
5,ch12_environment.csv,True,67
6,ch12_error_fix_example.md,True,1290
7,ch12_error_fix_prompt_template.md,True,1148
8,ch12_execution_gate.csv,True,499
9,ch12_generated_code_static_scan.csv,True,262



저장 위치: c:\dev\llm-data-analysis-course\chapter12\reports\ch12_20261006_123838_046374
저장 파일 수: 25개
현재 위험 예제의 최종 신뢰 판단: 사용 보류
Evidence 저장 완료 — 제한 실행과 사후 검증은 미수행 상태입니다.


### STEP 12. Evidence 저장과 최종 신뢰 범위 — 결과 확인

#### 결과 관찰

노트북에서 확인한 검증 결과와 기록을 Evidence 파일 25개로 저장했다.

| 확인 항목 | 결과 |
|---|---|
| 저장 파일 수 | 25개 |
| 파일 존재 여부 | 모두 True |
| 파일 크기 | 모두 0보다 큼 |
| 위험 예제의 최종 신뢰 판단 | 사용 보류 |
| 제한 실행 | NOT_RUN |
| 사후 검증 | NOT_EVALUATED |

저장 폴더는 chapter12/reports/ch12_20261006_123838_046374이다.

#### 나의 해석과 판단

검증 결과표와 판단 근거를 파일로 보존했다. 다만 파일 저장 성공은 코드 실행 승인이나 모든 검증의 완료를 의미하지 않는다.

현재 공식 위험 예제에는 실행 차단 사유가 남아 있으며, 사람의 실행 승인도 없다. 따라서 최종 신뢰 판단을 사용 보류로 유지했다.

#### Evidence

파일 목록에서 25개 파일의 존재 여부와 크기를 확인했다.

저장 자료에는 입력 파일의 크기·SHA256, 데이터 범위, PK/FK 검사, 집계 결과, 정적 스캔, Feature 검토, Execution Gate, 사람 검토 초안, 실행 보류 및 사후 미평가 기록이 포함됐다.

입력 파일의 SHA256은 이번 저장 시점의 기록이며, 실행 전후 원본 보존을 비교·검증한 결과는 아니다.

#### 수정·저장 내용

실행 시각으로 구분한 새 폴더를 만들어 결과표, 검사 대상 코드 문자열, 오류 요청 템플릿과 예시, 최종 요약을 저장했다.

검사 대상 코드는 실행하지 않고 텍스트 파일로 보관했다. Evidence는 노트북 결과를 직접 저장한 것이며, 공식 일괄 보고서 생성 스크립트의 출력은 아니다.

#### 최종 신뢰 범위와 한계

확인된 범위는 검증용 복사본의 필수 컬럼·PK/FK 관계와, 날짜가 있는 completed 주문의 금액 계산·merge·집계 총합 일치다.

기준·카테고리·월별 총합은 모두 138,985,000이었다.

제외한 행의 영향, 2027-01 날짜의 타당성, Feature의 예측 시점 가용성, Sandbox 설정, 사람의 최종 판단은 추가 확인이 필요하다.

현재 위험 예제는 사용 보류 상태이며, 승인된 코드의 제한 실행과 실행 후 검증은 아직 수행하지 않았다.